# BrownDye2 preparation: protein-ligand complex

0. Validate complex (chain IDs, SMILES)
1. Fix protein with PDBFixer, assign ligand topology with RDKit
2. Assemble complex PDB for tleap
3. AmberTools parameterization (antechamber, parmchk2, tleap) → prmtop/rst7
4. ParmEd convert to complex.pqr
5. APBS input generation using pdb2pqr inputgen
6. Run APBS

Steps 0-2 in this notebook, steps 3-6 via `scripts/browndye/run_amber_apbs.sh`.

In [ ]:
import copy
from pathlib import Path

from Bio.Data.IUPACData import protein_letters_3to1
from Bio.PDB import PDBIO, PDBParser, Select
from rdkit import Chem

from mdpp.prep.ligand import assign_topology
from mdpp.prep.protein import fix_pdb

# User configuration
COMPLEX_PDB = Path("TurboID-bioAMP_model_0.pdb")
WORKDIR = Path("tmp")
WORKDIR.mkdir(exist_ok=True, parents=True)

# Required: canonical SMILES of the ligand
LIGAND_SMILES = r"Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P]([O-])(=O)OC(=O)CCCC[C@@H]4SC[C@@H]5NC(=O)N[C@H]45)[C@@H](O)[C@H]3O"
PROTEIN_CHAIN_ID = "A"
LIGAND_CHAIN_ID = "B"
PH = 7.4

## Step 0: Validate complex

In [ ]:
STANDARD_AA = {code.upper() for code in protein_letters_3to1}

parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", str(COMPLEX_PDB))
model = structure[0]

chains = {chain.id: chain for chain in model}
assert PROTEIN_CHAIN_ID in chains, f"Chain {PROTEIN_CHAIN_ID} not found. Available: {list(chains)}"
assert LIGAND_CHAIN_ID in chains, f"Chain {LIGAND_CHAIN_ID} not found. Available: {list(chains)}"

# Summarize selected chains
prot_chain = chains[PROTEIN_CHAIN_ID]
prot_resnames = {
    res.get_resname().strip()
    for res in prot_chain.get_residues()
    if res.get_resname().strip() not in ("HOH", "WAT")
}
n_prot_res = sum(1 for _ in prot_chain.get_residues())
non_standard = prot_resnames - STANDARD_AA
if non_standard:
    print(f"WARNING: chain {PROTEIN_CHAIN_ID} has non-standard residues: {non_standard}")
print(f"Protein chain {PROTEIN_CHAIN_ID}: {n_prot_res} residues")

lig_chain = chains[LIGAND_CHAIN_ID]
lig_resnames = {res.get_resname().strip() for res in lig_chain.get_residues()}
assert len(lig_resnames) == 1, f"Ligand chain {LIGAND_CHAIN_ID} has {len(lig_resnames)} residues"
LIG_RESNAME = next(iter(lig_resnames))
n_lig_atoms = sum(1 for _ in lig_chain.get_atoms())
print(f"Ligand chain {LIGAND_CHAIN_ID}: residue(s) {lig_resnames}, {n_lig_atoms} atoms")

## Step 1a: Fix protein

In [ ]:
class _ChainSelect(Select):
    """Select residues in one or more chains."""

    def __init__(self, chain_ids: str | list[str]) -> None:
        self.chain_ids = {chain_ids} if isinstance(chain_ids, str) else set(chain_ids)

    def accept_chain(self, chain) -> int:
        return int(chain.id in self.chain_ids)


pdb_io = PDBIO()
pdb_io.set_structure(structure)

# Extract protein chain
protein_pdb = WORKDIR / "protein.pdb"
pdb_io.save(str(protein_pdb), _ChainSelect(PROTEIN_CHAIN_ID))
print(f"Extracted protein chain {PROTEIN_CHAIN_ID} -> {protein_pdb}")

# Fix protein (add missing residues, atoms, hydrogens)
protein_fixed_pdb = WORKDIR / "protein_fixed.pdb"
fix_pdb(protein_pdb, protein_fixed_pdb, pH=PH)
print(f"Fixed protein -> {protein_fixed_pdb}")

## Step 1b: Assign ligand topology

In [ ]:
# Extract ligand chain
ligand_pdb = WORKDIR / "ligand.pdb"
pdb_io.save(str(ligand_pdb), _ChainSelect(LIGAND_CHAIN_ID))
print(f"Extracted ligand chain {LIGAND_CHAIN_ID} -> {ligand_pdb}")

# Validate SMILES and compute net charge
template_mol = Chem.MolFromSmiles(LIGAND_SMILES, sanitize=True)
assert template_mol is not None, f"Invalid SMILES: {LIGAND_SMILES}"
ligand_net_charge = Chem.GetFormalCharge(template_mol)
print(f"SMILES: {Chem.MolToSmiles(template_mol)}")
print(f"Net charge: {ligand_net_charge}")

# Assign bond orders from SMILES template
mol = Chem.MolFromPDBFile(str(ligand_pdb), sanitize=True, removeHs=True)
assert mol is not None, f"RDKit failed to parse {ligand_pdb}"
mol_assigned = assign_topology(mol, template_mol)

# Set molecule name
mol_assigned.SetProp("_Name", LIG_RESNAME)

# Write SDF for antechamber
ligand_sdf = WORKDIR / "ligand.sdf"
with Chem.SDWriter(str(ligand_sdf)) as w:
    w.write(mol_assigned)
print(f"Wrote topology-assigned ligand -> {ligand_sdf}")

## Step 2: Assemble complex PDB for tleap

In [ ]:
# Re-parse fixed protein and graft the ligand chain onto it
fixed_struct = parser.get_structure("fixed", str(protein_fixed_pdb))
fixed_model = fixed_struct[0]

# Add ligand chain from original structure
lig_chain_copy = copy.deepcopy(chains[LIGAND_CHAIN_ID])
fixed_model.add(lig_chain_copy)

# Write combined complex
complex_pdb = WORKDIR / "complex.pdb"
pdb_io.set_structure(fixed_struct)
pdb_io.save(str(complex_pdb), _ChainSelect([PROTEIN_CHAIN_ID, LIGAND_CHAIN_ID]))
print(f"Assembled complex -> {complex_pdb}")

## Steps 3-6: AmberTools, PQR, APBS

Edit the constants at the top of `scripts/browndye/run_amber_apbs.sh`, then run:

```bash
bash scripts/browndye/run_amber_apbs.sh
```

3. AmberTools parameterization (antechamber, parmchk2, tleap) → prmtop/rst7
4. ParmEd convert to complex.pqr
5. APBS input generation using pdb2pqr inputgen
6. Run APBS